# 🔥 Fitness AI — Calorie Burn Prediction Model Training

This notebook trains **4 Machine Learning models** to predict calories burned during exercise:

| # | Model | Type |
|---|---|---|
| 1 | 🌲 Random Forest | Ensemble (Bagging) |
| 2 | ⚡ XGBoost | Ensemble (Boosting) |
| 3 | 🔷 SVM (SVR) | Kernel-based |
| 4 | 📈 Linear Regression | Polynomial Features |

**Features used:**  `exercise_type`, `intensity` (reps/sec), `body_weight_kg`, `duration_secs`, `reps`  
**Target:** `calories_burned` (kcal)

---

## 📦 Step 1 — Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

print('✅ All libraries imported successfully!')

## 🏋️ Step 2 — Generate Synthetic Dataset

We generate **10,000 synthetic workout samples** using the **MET (Metabolic Equivalent of Task)** formula:

$$\text{Calories} = \text{MET} \times \text{Weight (kg)} \times \text{Duration (hours)}$$

The MET value is adjusted by the exercise intensity (reps per minute) to make it realistic.

In [ ]:
def generate_synthetic_data(num_samples=10000):
    np.random.seed(42)
    exercises = ['Left Bicep Curl', 'Right Bicep Curl', 'Squat', 'Sit-up']
    base_mets = {
        'Left Bicep Curl':  3.0,
        'Right Bicep Curl': 3.0,
        'Squat':            5.0,
        'Sit-up':           3.8
    }

    data = []
    for _ in range(num_samples):
        exercise       = np.random.choice(exercises)
        body_weight_kg = np.round(np.random.uniform(50.0, 120.0), 1)
        duration_secs  = np.round(np.random.uniform(10.0, 1800.0), 1)
        reps           = np.random.randint(1, 500)

        intensity_val  = round(reps / duration_secs, 4)          # reps/sec
        duration_hours = duration_secs / 3600.0
        rpm            = reps / (duration_secs / 60.0) if duration_secs > 0 else 0

        intensity_modifier = max(0.5, min(rpm / 20.0, 2.0))      # clamp [0.5, 2.0]
        adjusted_met       = base_mets[exercise] * intensity_modifier
        calories_burned    = adjusted_met * body_weight_kg * duration_hours
        calories_burned   += np.random.normal(0, 0.2)            # tiny noise
        calories_burned    = max(1.0, calories_burned)

        data.append({
            'exercise_type':  exercise,
            'intensity':      intensity_val,
            'body_weight_kg': body_weight_kg,
            'duration_secs':  duration_secs,
            'reps':           reps,
            'calories_burned': np.round(calories_burned, 2)
        })

    return pd.DataFrame(data)


df = generate_synthetic_data(num_samples=10000)
print(f'✅ Dataset generated: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

## 📊 Step 3 — Explore the Dataset

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Calories distribution
axes[0].hist(df['calories_burned'], bins=50, color='coral', edgecolor='white')
axes[0].set_title('Distribution of Calories Burned', fontsize=13)
axes[0].set_xlabel('Calories (kcal)')
axes[0].set_ylabel('Count')

# Avg calories per exercise
avg_cal = df.groupby('exercise_type')['calories_burned'].mean().sort_values()
axes[1].barh(avg_cal.index, avg_cal.values, color=['#4CAF50','#2196F3','#FF9800','#9C27B0'])
axes[1].set_title('Average Calories Burned by Exercise', fontsize=13)
axes[1].set_xlabel('Avg Calories (kcal)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(7, 5))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 💾 Step 4 — Save Dataset to CSV

In [ ]:
dataset_dir  = os.path.join(os.getcwd(), 'datasets')
os.makedirs(dataset_dir, exist_ok=True)
dataset_path = os.path.join(dataset_dir, 'fitness_tracker_dataset.csv')
df.to_csv(dataset_path, index=False)
print(f'✅ Dataset saved → {dataset_path}')

## ✂️ Step 5 — Prepare Features & Train/Test Split

In [ ]:
X = df[['exercise_type', 'intensity', 'body_weight_kg', 'duration_secs', 'reps']]
y = df['calories_burned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## ⚙️ Step 6 — Build the Shared Preprocessor

- **Numeric features** → `StandardScaler` (zero mean, unit variance) — required by SVM
- **Categorical features** → `OneHotEncoder` (exercise type → binary columns)

In [ ]:
numeric_features     = ['intensity', 'body_weight_kg', 'duration_secs', 'reps']
categorical_features = ['exercise_type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                           numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'),     categorical_features)
])

print('✅ Preprocessor ready!')

## 🌲 Step 7 — Train Random Forest Regressor

An **ensemble of 100 decision trees** that average their predictions (bagging). Robust to outliers and works well without feature scaling.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor',    RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

rf_r2  = r2_score(y_test, rf_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_mse = mean_squared_error(y_test, rf_preds)

print(f'🌲 Random Forest  →  R²: {rf_r2:.4f}  |  MAE: {rf_mae:.3f}  |  RMSE: {np.sqrt(rf_mse):.3f}')

## ⚡ Step 8 — Train XGBoost Regressor

**Gradient Boosting** — builds trees sequentially, each correcting the errors of the previous one. One of the best algorithms on tabular data.

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor',    XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)

xgb_r2  = r2_score(y_test, xgb_preds)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_mse = mean_squared_error(y_test, xgb_preds)

print(f'⚡ XGBoost         →  R²: {xgb_r2:.4f}  |  MAE: {xgb_mae:.3f}  |  RMSE: {np.sqrt(xgb_mse):.3f}')

## 🔷 Step 9 — Train SVM (Support Vector Regressor)

Uses an **RBF (Gaussian) kernel** to map data into a high-dimensional space and find the best-fit hyperplane. Excellent at capturing non-linear relationships. Requires scaled features.

In [ ]:
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor',    SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))
])

svm_pipeline.fit(X_train, y_train)
svm_preds = svm_pipeline.predict(X_test)

svm_r2  = r2_score(y_test, svm_preds)
svm_mae = mean_absolute_error(y_test, svm_preds)
svm_mse = mean_squared_error(y_test, svm_preds)

print(f'🔷 SVM (SVR)       →  R²: {svm_r2:.4f}  |  MAE: {svm_mae:.3f}  |  RMSE: {np.sqrt(svm_mse):.3f}')

## 📈 Step 10 — Train Linear Regression (Polynomial Features)

Extends standard linear regression by adding **degree-2 polynomial** feature combinations, allowing the model to learn curved relationships.

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly',         PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor',    LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

lr_r2  = r2_score(y_test, lr_preds)
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_mse = mean_squared_error(y_test, lr_preds)

print(f'📈 Linear Reg.     →  R²: {lr_r2:.4f}  |  MAE: {lr_mae:.3f}  |  RMSE: {np.sqrt(lr_mse):.3f}')

## 📊 Step 11 — Compare All Models

In [ ]:
results = pd.DataFrame({
    'Model':  ['🌲 Random Forest', '⚡ XGBoost', '🔷 SVM (SVR)', '📈 Linear Regression'],
    'R²':     [rf_r2,  xgb_r2,  svm_r2,  lr_r2],
    'MAE':    [rf_mae, xgb_mae, svm_mae, lr_mae],
    'RMSE':   [np.sqrt(rf_mse), np.sqrt(xgb_mse), np.sqrt(svm_mse), np.sqrt(lr_mse)]
}).sort_values('R²', ascending=False).reset_index(drop=True)

print('========== Model Performance Summary ==========')
print(results.to_string(index=False))
results

In [ ]:
# Bar chart — R² scores
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#4CAF50', '#FF9800', '#2196F3', '#9C27B0']
model_names = ['Random\nForest', 'XGBoost', 'SVM\n(SVR)', 'Linear\nRegression']
r2_vals  = [rf_r2, xgb_r2, svm_r2, lr_r2]
mae_vals = [rf_mae, xgb_mae, svm_mae, lr_mae]

bars1 = axes[0].bar(model_names, r2_vals, color=colors, edgecolor='white', width=0.5)
axes[0].set_ylim(0.8, 1.01)
axes[0].set_title('R² Score (higher = better)', fontsize=12)
axes[0].set_ylabel('R²')
for bar, val in zip(bars1, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(model_names, mae_vals, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('MAE — Mean Absolute Error (lower = better)', fontsize=12)
axes[1].set_ylabel('MAE (kcal)')
for bar, val in zip(bars2, mae_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison — Calorie Burn Prediction', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter for each model
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

preds_list = [rf_preds, xgb_preds, svm_preds, lr_preds]
titles = ['🌲 Random Forest', '⚡ XGBoost', '🔷 SVM (SVR)', '📈 Linear Regression']

for ax, preds, title in zip(axes, preds_list, titles):
    ax.scatter(y_test, preds, alpha=0.3, s=6, color='steelblue')
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Actual (kcal)')
    ax.set_ylabel('Predicted (kcal)')
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted Calories', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 Step 12 — Save All 4 Models as `.pkl` Files

In [ ]:
models_dir = os.getcwd()   # saves to backend/ folder (same as main.py expects)

joblib.dump(rf_pipeline,  os.path.join(models_dir, 'rf_model.pkl'))
joblib.dump(xgb_pipeline, os.path.join(models_dir, 'xgb_model.pkl'))
joblib.dump(svm_pipeline, os.path.join(models_dir, 'svm_model.pkl'))
joblib.dump(lr_pipeline,  os.path.join(models_dir, 'lr_model.pkl'))

print('✅  rf_model.pkl   saved!')
print('✅  xgb_model.pkl  saved!')
print('✅  svm_model.pkl  saved!')
print('✅  lr_model.pkl   saved!')
print(f'\n📂 Location: {models_dir}')

---
## ✅ Done!

All 4 models are saved. The **FastAPI backend** (`main.py`) will automatically pick them up on the next run.

You can select the model from the **AI Calorie Predictor** modal in the frontend:
- 🌲 **Random Forest** — highest accuracy, great default choice
- ⚡ **XGBoost** — equally accurate, faster inference
- 🔷 **SVM** — excellent non-linear fit with RBF kernel
- 📈 **Linear Regression** — simplest and most interpretable